# Notebook 07: Isolated Model Training - Hospital C (General/Nephrology Focus)

## Objective
Train a local model exclusively on **Hospital C** dataset.
Compare performance of isolated models A, B, and C on a unified global test set.



In [1]:
import os
import sys
# Ensure project root is in sys.path for backend and scripts imports
root_path = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath('.')
if root_path not in sys.path:
    sys.path.insert(0, root_path)
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

from backend.ml.model import DiabetesRiskModel

ha = pd.read_csv(os.path.join(root_path, "data", "hospitals", "hospital_A.csv"))
hb = pd.read_csv(os.path.join(root_path, "data", "hospitals", "hospital_B.csv"))
hc = pd.read_csv(os.path.join(root_path, "data", "hospitals", "hospital_C.csv"))

feature_cols = ['age', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'bmi', 'glucose', 'hba1c', 'cholesterol', 'creatinine']
target_col = 'diabetes'

scaler = StandardScaler()
scaler.fit(pd.concat([ha, hb, hc])[feature_cols])

X_c = scaler.transform(hc[feature_cols].values)
y_c = hc[target_col].values

split_idx = int(len(X_c) * 0.8)
X_c_train, X_c_test = X_c[:split_idx], X_c[split_idx:]
y_c_train, y_c_test = y_c[:split_idx], y_c[split_idx:]

train_loader = DataLoader(TensorDataset(torch.tensor(X_c_train, dtype=torch.float32), torch.tensor(y_c_train, dtype=torch.float32)), batch_size=32, shuffle=True)

model_c = DiabetesRiskModel(input_dim=len(feature_cols))
optimizer = optim.Adam(model_c.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss()

model_c.train()
for epoch in range(25):
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(model_c(bx).squeeze(), by)
        loss.backward()
        optimizer.step()

print("Hospital C Local Training Complete.")


C:\Users\bahad\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Hospital C Local Training Complete.
